# 폐기물 이미지 분류 — 평가용 노트북

직접 학습한 DeepCNN의 최종 가중치(`final_trash_cnn.pt`)를 불러와 새로운 폐기물 사진을 분류합니다. **데이터셋 다운로드 및 재학습은 필요하지 않습니다.** CPU 또는 GPU에서 실행할 수 있습니다.

### 실행 방법
1. Google Colab에서 이 노트북을 엽니다.
2. 아래 1~2번 코드 셀을 순서대로 실행합니다. 가중치는 GitHub에서 다운로드하며, 다운로드가 실패하면 직접 업로드할 수 있습니다.
3. 3번 코드 셀을 실행하고 테스트할 이미지 파일(여러 장 선택 가능)을 업로드합니다. 다른 사진은 3번 셀만 다시 실행하면 됩니다.

모델은 12개 클래스 중 하나를 예측합니다. 최고 Softmax 점수가 0.8 미만이면 수동 검수 대상으로 표시합니다. 이 기준은 시범 운영용이며, 높은 점수에서도 오분류가 가능하고 여러 종류의 폐기물이 함께 있는 이미지에는 한계가 있습니다.


## 1. 실행 환경 및 모델 구조

In [ ]:
import io
from pathlib import Path
from urllib.request import urlretrieve

import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image, UnidentifiedImageError
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

class DeepCNN(nn.Module):
    def __init__(self, num_classes=12):
        super().__init__()
        layers = []
        in_channels = 3

        for out_channels, num_convs in [(32, 1), (64, 2), (128, 2), (256, 2)]:
            for _ in range(num_convs):
                layers.extend([
                    nn.Conv2d(in_channels, out_channels, 3, padding=1),
                    nn.BatchNorm2d(out_channels),
                    nn.ReLU()
                ])
                in_channels = out_channels
            layers.append(nn.MaxPool2d(2))

        layers.append(nn.AdaptiveAvgPool2d((4, 4)))
        self.features = nn.Sequential(*layers)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))


## 2. 최종 가중치 불러오기

GitHub의 `main` 브랜치 루트에 가중치 파일이 등록되어 있어야 자동 다운로드됩니다. 파일이 없는 경우 직접 업로드할 수 있습니다. 저장소 또는 파일 위치를 변경했다면 아래 `MODEL_URL`을 실제 raw URL로 수정합니다.

In [ ]:
MODEL_URL = "https://raw.githubusercontent.com/hyo08009-bit/-/main/final_trash_cnn.pt"
MODEL_PATH = Path("final_trash_cnn.pt")

if not MODEL_PATH.is_file():
    tmp_path = MODEL_PATH.with_suffix(".download")
    try:
        urlretrieve(MODEL_URL, str(tmp_path))
        tmp_path.replace(MODEL_PATH)
        print("Downloaded from GitHub")
    except Exception as exc:
        tmp_path.unlink(missing_ok=True)
        print("GitHub download unavailable:", exc)
        print("Upload final_trash_cnn.pt manually.")
        from google.colab import files
        uploaded_weights = files.upload()
        if MODEL_PATH.name not in uploaded_weights:
            raise FileNotFoundError("Please upload final_trash_cnn.pt and rerun this cell.")
        MODEL_PATH.write_bytes(uploaded_weights[MODEL_PATH.name])

checkpoint = torch.load(MODEL_PATH, map_location="cpu", weights_only=True)
assert checkpoint["architecture"] == "DeepCNN"
class_names = list(checkpoint["class_names"])
assert len(class_names) == checkpoint["num_classes"]
model = DeepCNN(checkpoint["num_classes"]).to(device)
model.load_state_dict(checkpoint["model_state_dict"], strict=True)
model.eval()

image_size = checkpoint["image_size"]
inference_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(checkpoint["mean"], checkpoint["std"])
])

print("Model loaded successfully")
print("Classes:", class_names)
print("Image size:", image_size)


## 3. 이미지 업로드 및 분류

이미지마다 상위 3개 예측과 Softmax 점수를 표시하고, 수동 검수 여부를 안내합니다. 이 셀을 반복 실행해 새로운 이미지를 테스트할 수 있습니다.

In [ ]:
from google.colab import files

THRESHOLD = 0.8

def predict_image(image):
    x = inference_transform(image.convert("RGB")).unsqueeze(0).to(device)
    with torch.inference_mode():
        probs = torch.softmax(model(x), dim=1)
        scores, indices = torch.topk(probs[0], k=3)
    return [(class_names[i], s) for i, s in zip(indices.cpu().tolist(), scores.cpu().tolist())]

uploaded = files.upload()

for filename, content in uploaded.items():
    try:
        with Image.open(io.BytesIO(content)) as source:
            image = source.convert("RGB")
    except (UnidentifiedImageError, OSError):
        print(f"Skipped: {filename} (not a valid image)")
        continue

    predictions = predict_image(image)
    prediction, confidence = predictions[0]
    decision = "Automatic classification" if confidence >= THRESHOLD else "Manual inspection required"

    plt.figure(figsize=(5, 5))
    plt.imshow(image)
    plt.title(f"{prediction} ({confidence:.2%})")
    plt.axis("off")
    plt.show()

    print("File:", filename)
    for name, score in predictions:
        print(f"{name}: {score:.2%}")
    print("Decision:", decision)
    print("-" * 40)
